In [1]:
import pandas as pd
import numpy as np
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.14.0+cpu
CUDA available: False


In [2]:
columns = [
    "id",
    "label",
    "statement",
    "subject",
    "speaker",
    "speaker_job",
    "state",
    "party",
    "barely_true",
    "false",
    "half_true",
    "mostly_true",
    "pants_fire",
    "context"
]

In [3]:
train = pd.read_csv(
    "../dataset/liar/train.tsv",
    sep="\t",
    header=None,
    names=columns
)

valid = pd.read_csv(
    "../dataset/liar/valid.tsv",
    sep="\t",
    header=None,
    names=columns
)

test = pd.read_csv(
    "../dataset/liar/test.tsv",
    sep="\t",
    header=None,
    names=columns
)

In [4]:
print("Train:", train.shape)
print("Validation:", valid.shape)
print("Test:", test.shape)

train.head()

Train: (10240, 14)
Validation: (1284, 14)
Test: (1267, 14)


,id,label,statement,subject,speaker,speaker_job,state,party,barely_true,false,half_true,mostly_true,pants_fire,context
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver
3,1123.json,false,Health care reform legislation is likely to ma...,health-care,blog-posting,NaN,NaN,none,7.0,19.0,3.0,5.0,44.0,a news release
4,9028.json,half-true,The economic turnaround started at the end of ...,"economy,jobs",charlie-crist,NaN,Florida,democrat,15.0,9.0,20.0,19.0,2.0,an interview on CNN


In [5]:
train_bert = train[["statement", "label"]].copy()
valid_bert = valid[["statement", "label"]].copy()
test_bert = test[["statement", "label"]].copy()

In [6]:
train_bert = train_bert.dropna(subset=["statement", "label"])
valid_bert = valid_bert.dropna(subset=["statement", "label"])
test_bert = test_bert.dropna(subset=["statement", "label"])

In [7]:
train_bert = train_bert.drop_duplicates()
valid_bert = valid_bert.drop_duplicates()
test_bert = test_bert.drop_duplicates()

In [8]:
print("Train:", train_bert.shape)
print("Validation:", valid_bert.shape)
print("Test:", test_bert.shape)

Train: (10229, 2)
Validation: (1284, 2)
Test: (1267, 2)


In [9]:
label2id = {
    "barely-true": 0,
    "false": 1,
    "half-true": 2,
    "mostly-true": 3,
    "pants-fire": 4,
    "true": 5
}

id2label = {
    0: "barely-true",
    1: "false",
    2: "half-true",
    3: "mostly-true",
    4: "pants-fire",
    5: "true"
}

train_bert["label_id"] = train_bert["label"].map(label2id)
valid_bert["label_id"] = valid_bert["label"].map(label2id)
test_bert["label_id"] = test_bert["label"].map(label2id)

In [10]:
train_bert[["statement", "label", "label_id"]].head()

,statement,label,label_id
0,Says the Annies List political group supports ...,false,1
1,When did the decline of coal start? It started...,half-true,2
2,"Hillary Clinton agrees with John McCain ""by vo...",mostly-true,3
3,Health care reform legislation is likely to ma...,false,1
4,The economic turnaround started at the end of ...,half-true,2


In [11]:
print(train_bert["label_id"].isna().sum())
print(train_bert["label_id"].value_counts().sort_index())

0
label_id
0    1654
1    1988
2    2112
3    1962
4     839
5    1674
Name: count, dtype: int64


In [12]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_bert[["statement", "label_id"]].reset_index(drop=True)
)

valid_dataset = Dataset.from_pandas(
    valid_bert[["statement", "label_id"]].reset_index(drop=True)
)

test_dataset = Dataset.from_pandas(
    test_bert[["statement", "label_id"]].reset_index(drop=True)
)

print(train_dataset)
print(valid_dataset)
print(test_dataset)

Dataset({
    features: ['statement', 'label_id'],
    num_rows: 10229
})
Dataset({
    features: ['statement', 'label_id'],
    num_rows: 1284
})
Dataset({
    features: ['statement', 'label_id'],
    num_rows: 1267
})


In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [14]:
sample_text = train_bert["statement"].iloc[0]

tokens = tokenizer(sample_text)

print("Original:")
print(sample_text)

print("\nToken IDs:")
print(tokens["input_ids"])

print("\nAttention Mask:")
print(tokens["attention_mask"])

Original:
Says the Annies List political group supports third-trimester abortions on demand.

Token IDs:
[101, 2758, 1996, 8194, 2015, 2862, 2576, 2177, 6753, 2353, 1011, 12241, 20367, 11324, 2015, 2006, 5157, 1012, 102]

Attention Mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [17]:
def tokenize_function(batch):
    return tokenizer(
        batch["statement"],
        truncation=True,
        max_length=128
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_valid = valid_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/10229 [00:00<?, ? examples/s]

Map:   0%|          | 0/1284 [00:00<?, ? examples/s]

Map:   0%|          | 0/1267 [00:00<?, ? examples/s]

In [25]:
tokenized_train = tokenized_train.rename_column(
    "label_id",
    "labels"
)

tokenized_valid = tokenized_valid.rename_column(
    "label_id",
    "labels"
)

tokenized_test = tokenized_test.rename_column(
    "label_id",
    "labels"
)

In [16]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=6,
    id2label=id2label,
    label2id=label2id
)

print(model.config.num_labels)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

c:\Users\gpshu\OneDrive\Desktop\Misinformation-Detection-System\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gpshu\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


6


In [27]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [28]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(
        labels,
        predictions,
        average="macro"
    )
    weighted_f1 = f1_score(
        labels,
        predictions,
        average="weighted"
    )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1
    }

In [29]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="../models/distilbert_check",

    num_train_epochs=1,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="steps",
    eval_steps=100,

    logging_steps=50,

    save_strategy="steps",
    save_steps=100,

    report_to="none"
)

In [30]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [31]:
from transformers import TrainingArguments, Trainer

small_train = tokenized_train.select(range(100))
small_valid = tokenized_valid.select(range(50))

small_training_args = TrainingArguments(
    output_dir="../models/distilbert_sanity_check",

    max_steps=5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    learning_rate=2e-5,
    weight_decay=0.01,

    logging_steps=1,

    eval_strategy="no",
    save_strategy="no",

    report_to="none"
)

small_trainer = Trainer(
    model=model,
    args=small_training_args,

    train_dataset=small_train,
    eval_dataset=small_valid,

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [32]:
train_result = small_trainer.train()

Step,Training Loss
1,1.844776
2,1.799124
3,1.770081
4,1.785489
5,1.798011
